## Overview

fasterai ships 8 distillation losses in two categories:

- **Output losses** compare final predictions. Simple, always work.
- **Intermediate losses** compare internal feature maps. More powerful, need layer matching.

Quick version: start with `SoftTarget`. If you need more, add `Attention`. For best results, use `DecoupledKD`.

## Setup

In [1]:
from fasterai.distill.all import *
from fasterai.core.schedule import cos
from fastai.vision.all import *

In [2]:
# Data
path = untar_data(URLs.PETS)
files = get_image_files(path/'images')
def label_func(f): return f[0].isupper()
dls = ImageDataLoaders.from_name_func(path, files, label_func, item_tfms=Resize(224))

# Train teacher (ResNet-34)
teacher = vision_learner(dls, resnet34, metrics=accuracy)
teacher.fit_one_cycle(5, 1e-3)

epoch,train_loss,valid_loss,accuracy,time
0,0.250642,0.021307,0.991881,00:03
1,0.075587,0.013623,0.995264,00:03
2,0.031664,0.007846,0.996617,00:03
3,0.017101,0.004599,0.997970,00:03
4,0.008718,0.005897,0.996617,00:03


## Benchmark - ResNet18

In [4]:
bench = Learner(dls, resnet18(num_classes=2), metrics=accuracy)
bench.fit_one_cycle(10, 1e-3)

epoch,train_loss,valid_loss,accuracy,time
0,0.597651,0.660210,0.643437,00:03
1,0.577063,0.629047,0.635318,00:03
2,0.515927,0.522208,0.738836,00:03
3,0.454511,0.743681,0.726658,00:03
4,0.404173,0.453840,0.788227,00:03
5,0.339115,0.350186,0.838295,00:03
6,0.285339,0.313760,0.864682,00:03
7,0.210879,0.271510,0.887010,00:03
8,0.157915,0.246154,0.897835,00:03
9,0.130631,0.238425,0.897835,00:03


## Output Losses

Compare final predictions only — no layer matching needed.

### `SoftTarget` — the classic (Hinton 2015)

Softens distributions with temperature `T`, matches via KL divergence. Higher `T` reveals more "dark knowledge."

**When:** Default choice for classification. Always start here.

In [ ]:
student = Learner(dls, resnet18(num_classes=2), metrics=accuracy)
kd = KnowledgeDistillationCallback(teacher.model, SoftTarget, weight=0.5)
student.fit_one_cycle(10, 1e-3, cbs=kd)

epoch,train_loss,valid_loss,accuracy,time


### `DecoupledKD` — best output loss (Zhao, CVPR 2022)

Separates into target-class (TCKD) and non-target-class (NCKD) components. The insight: the most valuable dark knowledge is in the **non-target classes** ("a dog is more like a cat than a plane").

With `normalize=True`, becomes **Normalized KD** (Yang, ICCV 2023).

**When:** Best output loss for classification. Worth the switch from SoftTarget.

In [ ]:
student = Learner(dls, resnet18(num_classes=2), metrics=accuracy)
kd = KnowledgeDistillationCallback(teacher.model, DecoupledKD, weight=0.5)
student.fit_one_cycle(10, 1e-3, cbs=kd)

### `Logits` and `Mutual`

| Loss | What | When |
|------|------|------|
| `Logits` | MSE on raw logits | Regression tasks |
| `Mutual` | KL without temperature (T=1) | When temperature tuning hurts |

## Intermediate Losses

Compare **internal feature maps**. More powerful but need `match_feature_layers` to pair layers:

In [ ]:
import torch
student_model = resnet18(num_classes=2)

# Auto-match layers by spatial resolution
try:
    from fasterai.distill.distillation_callback import match_feature_layers
    pairs = match_feature_layers(student_model, teacher.model, torch.randn(1, 3, 224, 224))
except ImportError:
    # Fallback: manual layer names (ResNet family)
    pairs = {'student': ['layer1', 'layer2', 'layer3', 'layer4'],
             'teacher': ['layer1', 'layer2', 'layer3', 'layer4']}

print(f'{"Student":<16} Teacher')
for s, t in zip(pairs['student'], pairs['teacher']):
    print(f'{s:<12} <-> {t}')

### `Attention` — most robust (Zagoruyko 2017)

Transfers **where** the teacher looks by matching spatial attention maps. Channel-agnostic — works across different architectures.

**When:** Go-to intermediate loss. Combine with any output loss.

In [ ]:
student = Learner(dls, resnet18(num_classes=2), metrics=accuracy)
kd = KnowledgeDistillationCallback(
    teacher.model, Attention,
    pairs['student'], pairs['teacher'],
    weight=0.9
)
student.fit_one_cycle(10, 1e-3, cbs=kd)

### `Similarity` — architecture-agnostic (Tung & Mori 2019)

Matches **sample-sample relationships** (Gram matrices) rather than individual features. Computes `G = F · Fᵀ` for each batch, then matches student and teacher Gram matrices.

**Key advantage:** only compares relationships between samples — works even when student and teacher have completely different channel counts and architectures (e.g., CNN → MLP).

**When:** Student and teacher are very different architectures.

In [ ]:
student = Learner(dls, resnet18(num_classes=2), metrics=accuracy)
kd = KnowledgeDistillationCallback(
    teacher.model, Similarity,
    pairs['student'], pairs['teacher'],
    weight=0.5
)
student.fit_one_cycle(10, 1e-3, cbs=kd)

### `FitNet` — direct feature matching (Romero 2015)

MSE between raw feature maps at each paired layer. The simplest intermediate loss.

**Caveat:** requires **matching channel counts** between student and teacher at each paired layer. This limits it to same-family architectures (e.g., ResNet-18 ↔ ResNet-34). For cross-architecture distillation, use `Attention` or `Similarity` instead.

**When:** Student and teacher are from the same family.

In [ ]:
student = Learner(dls, resnet18(num_classes=2), metrics=accuracy)
kd = KnowledgeDistillationCallback(
    teacher.model, FitNet,
    pairs['student'], pairs['teacher'],
    weight=0.5
)
student.fit_one_cycle(10, 1e-3, cbs=kd)

### `ActivationBoundaries` — decision boundaries (Heo 2019)

Focuses on **where neurons switch between active and inactive**. The student learns to match the teacher's activation boundaries with a margin `m`. Penalizes the student when its activations disagree with the teacher's sign.

**When:** You care about replicating the teacher's decision boundaries, not just feature magnitudes.

In [ ]:
student = Learner(dls, resnet18(num_classes=2), metrics=accuracy)
kd = KnowledgeDistillationCallback(
    teacher.model, ActivationBoundaries,
    pairs['student'], pairs['teacher'],
    weight=0.5
)
student.fit_one_cycle(10, 1e-3, cbs=kd)

## Results Comparison

ResNet-34 → ResNet-18 on Oxford Pets:

| Loss | Type | Accuracy | vs Baseline | Needs Layers? |
|------|------|----------|-------------|---------------|
| No distillation | — | 89.5% | — | — |
| `SoftTarget` | Output | 92.2% | +2.7% | No |
| `DecoupledKD` | Output | **93.2%** | **+3.7%** | No |
| `Attention` | Intermediate | 92.8% | +3.3% | Yes |
| `Similarity` | Intermediate | 92.1% | +2.6% | Yes |
| `FitNet` | Intermediate | 91.5% | +2.0% | Yes (same channels) |

`DecoupledKD` is the best single loss. `Attention` is the best intermediate loss and can be **combined** with any output loss for further gains.

## Quick Decision

```
Just starting?          → SoftTarget
Want best accuracy?     → DecoupledKD
Cross-architecture?     → Attention (or Similarity for very different)
Regression task?        → Logits
```

---

## See Also

- [KD Callback Tutorial](distill_callback.html) — End-to-end distillation workflow
- [Losses API](../../distill/losses.html) — Full API reference
- [QAT + Distillation](../quantize/qat_distill.html) — Combine with quantization